In [1]:
import tensorflow as tf

# give it perfect path of the data
train_dir = '/content/drive/MyDrive/Colab Notebooks/Training'
test_dir = '/content/drive/MyDrive/Colab Notebooks/Testing'

# 1. Training Dataset Load karein (Model ko sikhane ke liye)
train_dataset = tf.keras.utils.image_dataset_from_directory(
  train_dir,
  image_size=(224, 224),
  batch_size=32
)

# 2. Testing Dataset Load karein (Model ka 'Exam' lene ke liye)
test_dataset = tf.keras.utils.image_dataset_from_directory(
  test_dir,
  image_size=(224, 224),
  batch_size=32
)

# Optional but recommended for speed: Data ko cache karna
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Found 2600 files belonging to 4 classes.
Found 394 files belonging to 4 classes.


In [2]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

# 1. Data Augmentation (Thoda ghumana aur flip karna taaki model ratey nahi)
data_augmentation = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.1),
])

# 2. Pre-trained Dimaag (Base Model) load karein
base_model = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False, # Original dimaag ka aakhri hissa kaat diya
    weights='imagenet'
)
base_model.trainable = False # Isko freeze kar diya taaki purani knowledge destroy na ho

# 3. Apna custom Aarogyam ScanAI Model banayein
model = models.Sequential([
    data_augmentation,           # Pehle image augment hogi
    base_model,                  # Phir base model usme features dhoondhega
    layers.GlobalAveragePooling2D(), # Data ko flat/compact karega
    layers.Dense(256, activation='relu'), # Thodi aur processing (Hidden layer)
    layers.Dropout(0.3),         # Overfitting rokne ke liye 30% connections randomly drop karega
    layers.Dense(4, activation='softmax') # FINAL LAYER: 4 Classes (Bimariyan) hain, isliye 4.
])

# 4. Model ko Compile karein (Rules set karein)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Jab 2 se zyada classes hon toh yeh use hota hai
    metrics=['accuracy']
)

print("Model successfully ban gaya! Training start ho rahi hai...\n")

# 5. Training (Pardhai) Start Karein!
epochs = 10 # Model poore data ko 10 baar dekhega
history = model.fit(
    train_dataset,
    validation_data=test_dataset, # Har round ke baad test data par apna exam dega
    epochs=epochs
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model successfully ban gaya! Training start ho rahi hai...

Epoch 1/10
18/82 ━━━━━━━━━━━━━━━━━━━━ 3:11 3s/step - accuracy: 0.4054 - loss: 1.2451

KeyboardInterrupt: 

In [ ]:
epochs = 10
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=epochs
)

In [ ]:
# Model ko apne Google Drive mein save karein
model_path = '/content/drive/MyDrive/Aarogyam_ScanAI_Final.keras'
model.save(model_path)

print(f"Badhai ho! Aapka model yahan save ho gaya hai: {model_path}")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import os

# 1. ERROR FIX: Classes ke naam seedha Drive ke folder se padh rahe hain
train_dir = '/content/drive/MyDrive/Colab Notebooks/Training' # <-- Apne training folder ka sahi path yahan check kar lein
class_names = sorted([folder for folder in os.listdir(train_dir) if not folder.startswith('.')])
print(f"Aarogyam ScanAI Classes: {class_names}\n")

# 2. YAHAN CHANGE KAREIN: Apni kisi bhi nayi MRI image ka path daalein
test_image_path = '/content/drive/MyDrive/p (269).jpg'

# 3. Image ko load aur resize karein
img = image.load_img(test_image_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

# 4. Aarogyam ScanAI apna dimaag lagayega aur prediction dega
predictions = model.predict(img_array)
predicted_class = class_names[np.argmax(predictions)]
confidence = np.max(predictions) * 100

# 5. Screen par Asli Result aur Image Dikhayein
plt.imshow(img)
plt.title(f"Result: {predicted_class} ({confidence:.2f}%)")
plt.axis('off')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from google.colab import files
import io

# 1. Model Load Karein (Make sure aapka Drive mounted ho)
print("Aarogyam ScanAI Model Load ho raha hai...")
model_path = '/content/drive/MyDrive/Aarogyam_ScanAI_Final.keras' # Aapki save ki hui file
model = load_model(model_path)

# Classes ke naam (Make sure yeh wahi alphabetical order ho jo pehle print hua tha)
class_names = ['Glioma', 'Meningioma', 'No_Tumor', 'Pituitary']

# 2. Knowledge Base (Bimariyon ki details)
tumor_info = {
    'Glioma': {
        'definition': "Glioma brain ya spinal cord mein hone wala ek tumor hai. Yeh un cells (glial cells) mein shuru hota hai jo nerve cells ko support karte hain.",
        'causes': "Iska exact kaaran pata nahi hai, par age badhna aur radiation iske risk factors hain."
    },
    'Meningioma': {
        'definition': "Yeh tumor un jhilliyon (meninges) mein banta hai jo aapke brain aur spinal cord ko cover karti hain. Yeh usually bohot dheere badhta hai.",
        'causes': "Sir (head) par pehle kabhi radiation therapy hona ya kuch genetic disorders iske kaaran ho sakte hain."
    },
    'Pituitary': {
        'definition': "Yeh tumor brain ke base par maujood choti si Pituitary gland mein hota hai. Yeh body ke hormones ko disturb kar sakta hai.",
        'causes': "Pituitary gland mein cells out of control kyun badhte hain yeh abhi tak unknown hai. Kuch cases hereditary (khandani) hote hain."
    },
    'No_Tumor': {
        'definition': "Aapka MRI scan bilkul clear hai. Is scan mein koi bhi abnormal tumor detect nahi hua hai.",
        'causes': "Aap bilkul swasth (healthy) hain!"
    }
}

# 3. User se Image Upload karwayein
print("\n👇 Kripya test karne ke liye ek MRI image upload karein 👇")
uploaded = files.upload()

# 4. Image Process aur Prediction karein
for fn in uploaded.keys():
    print(f"\nAnalyzing '{fn}'...")

    # Image ko memory se read karein
    img_bytes = uploaded[fn]
    img = image.load_img(io.BytesIO(img_bytes), target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    # AI se result poochein
    predictions = model.predict(img_array, verbose=0)
    predicted_class = class_names[np.argmax(predictions)]
    confidence = np.max(predictions) * 100

    # Knowledge Base se details nikalein
    info = tumor_info.get(predicted_class)

    # 5. Result ko Sundar tarike se Print karein
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    print("-" * 50)
    print(f"🩺 AAROGYAM ScanAI REPORT")
    print("-" * 50)
    print(f"🟢 Diagnosis : {predicted_class}")
    print(f"🎯 Confidence: {confidence:.2f}%\n")
    print(f"📖 What is it?\n{info['definition']}\n")
    print(f"⚠️ Potential Causes:\n{info['causes']}")
    print("-" * 50)